# Fashion Demand Forecasting: Exploratory Analysis & ML Pipeline
**Predicting Clothing Sales and Retail Demand with Machine Learning**

This notebook demonstrates:
1. Dataset loading and integrity inspection
2. Data cleaning, cancellation filtering, and fashion taxonomy classification
3. Weekly demand aggregation and exploratory time-series analysis
4. Leakage-safe feature engineering (shifted lags, rolling statistics, calendar harmonics)
5. Chronological holdout validation: Naive baseline vs 4-Week Moving Average vs Random Forest Regressor
6. Multi-step recursive future demand forecasting with empirical prediction intervals.

In [ ]:
import sys
from pathlib import Path
# Set project root
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_raw_dataset, run_data_quality_report
from src.preprocessing import clean_retail_data, aggregate_demand_series
from src.feature_engineering import create_forecasting_features
from src.forecasting import DemandForecaster
from src.evaluation import compute_metrics

## 1. Data Ingestion and Integrity Check

In [ ]:
df_raw, meta = load_raw_dataset()
print(f"Loaded dataset: {meta.name}")
print(f"Total transactions: {meta.total_rows:,}")
print(f"Date Range: {meta.min_date} to {meta.max_date}")

quality_report = run_data_quality_report(df_raw)
print("Data quality summary:", quality_report)

## 2. Preprocessing & Fashion Category Classification

In [ ]:
df_clean, stats = clean_retail_data(df_raw, include_only_positive_demand=True)
print(f"Retained clean transactions: {stats.net_records_retained:,}")
print(f"Fashion coverage: {stats.fashion_coverage_pct:.1f}%")
print(f"Unique categories: {stats.unique_categories}")

# Display category transaction counts
df_clean['Category'].value_counts()

## 3. Demand Aggregation & Exploratory Analysis

In [ ]:
# Aggregate weekly demand for Tops
weekly_tops = aggregate_demand_series(df_clean, group_col='Category', group_val='Tops', freq='W-MON')
print(f"Observed weeks: {len(weekly_tops)}")

plt.figure(figsize=(12, 5))
plt.plot(weekly_tops['date'], weekly_tops['demand'], marker='o', label='Weekly Demand (Tops)')
plt.plot(weekly_tops['date'], weekly_tops['demand'].rolling(4).mean(), color='orange', linestyle='--', label='4-Week Moving Avg')
plt.title('Weekly Demand for Tops Over Time')
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 4. Leakage-Safe Feature Engineering

In [ ]:
featured = create_forecasting_features(weekly_tops, lags=[1, 2, 4, 8], rolling_windows=[4, 8], drop_na=True)
print(f"Constructed feature columns: {[c for c in featured.columns if c not in ['date', 'demand']]}")
featured.head()

## 5. Chronological Holdout Validation & Model Comparison

In [ ]:
models = ['naive', 'moving_average', 'random_forest']
results = []

for m in models:
    forecaster = DemandForecaster(model_type=m)
    metrics, holdout = forecaster.evaluate_holdout(weekly_tops, test_size=0.2)
    results.append({
        'Model': m,
        'MAE': metrics['MAE'],
        'RMSE': metrics['RMSE'],
        'WAPE (%)': metrics['WAPE'],
        'Test Periods': metrics['count']
    })

pd.DataFrame(results)

## 6. Multi-Step Future Forecast with Prediction Intervals

In [ ]:
rf_forecaster = DemandForecaster(model_type='random_forest')
rf_forecaster.fit(weekly_tops)

forecast_12w = rf_forecaster.predict_future(weekly_tops, horizon=12, freq='W-MON')
forecast_12w